# 11 - Ajuste de hiperparámetros
En este notebook explicamos cómo se han elegido los hiperparámetros de los modelos de la matriz de experimentos y qué valores han salido.

Se ajustan cuatro modelos: `logistic_regression`, `random_forest`, `xgboost_balanced` y `lightgbm`.

**Como regla principal:** el conjunto de test de una partición **nunca** se usa para elegir hiperparámetros. El ajuste de cada partición solo usa los datos de entrenamiento.

El ajuste de hiperparámetros no se hace en este notebook, ya se ha ejecutado. Se puede lanzar con `make tune` (`urban_accessibility_ml/modeling/tuning/tuning_run.py`) y sus resultados están en dos ficheros de `reports/results/`:
- `best_params.json`: los hiperparámetros elegidos en cada partición y modelo.
- `tuning-candidates.csv`: todas las candidatas que se probaron y su nota.

En este notebook mostramos cómo se ha hecho el ajuste y los resultados.

## Carga de datos y resultados del ajuste

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedGroupKFold

from urban_accessibility_ml.config import PROCESSED_DATA_DIR, REPORTS_DIR, SEED
from urban_accessibility_ml.modeling.tuning.tuning_constants import (
    SEARCH_SPACES,
    TUNING_FEATURE_SET,
    TUNING_INNER_SPLITS,
    TUNING_N_CANDIDATES,
    TUNING_PROFILE,
    TUNING_TIE_TOLERANCE,
)
from urban_accessibility_ml.modeling.tuning.tuning_results import load_best_params
from urban_accessibility_ml.modeling.train import run_one
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, use_style
from urban_accessibility_ml.preprocessing.features import get_xy
from urban_accessibility_ml.preprocessing.split import ID_COL, STREET_COL, build_all_splits

BEST_PARAMS_FILE = REPORTS_DIR / "results" / "best_params.json"
CANDIDATES_FILE = REPORTS_DIR / "results" / "tuning-candidates.csv"
TUNED_MODELS = list(SEARCH_SPACES)
PARTITION = ["strategy", "name", "fold"]
# One colour per model, the same as in notebooks 08 and 09
MODEL_COLORS = {
    "logistic_regression": SERIES[3],
    "random_forest": SERIES[0],
    "xgboost_balanced": SERIES[1],
    "lightgbm": SERIES[2],
}

use_style()

df = pd.read_csv(PROCESSED_DATA_DIR / "streets_unified.csv")
best_params = load_best_params(BEST_PARAMS_FILE)
candidates = pd.read_csv(CANDIDATES_FILE)
candidates["params"] = candidates["params"].map(json.loads)  # the params are stored as JSON text
print(f"{len(df)} tramos | {len(best_params)} particiones en best_params.json | {len(candidates)} candidatas en el CSV")

## Los modelos

| Modelo | Idea | Cómo compensa la clase A (muy poco frecuente) | ¿Se ajusta? |
|---|---|---|---|
| `dummy` | Predice siempre la clase más frecuente. Es el suelo que cualquier modelo debe superar. | - | no |
| `logistic_regression` | Un peso por variable, convertido en una probabilidad para cada clase. | `class_weight="balanced"` | sí (rejilla) |
| `random_forest` | Muchos árboles de decisión entrenados con muestras distintas, se promedian sus predicciones. | `class_weight="balanced"` | sí (aleatoria) |
| `xgboost_balanced` | Árboles construidos uno detrás de otro, cada uno corrige los errores de los anteriores (*boosting*). | pesos por tramo (`sample_weight`), que el modelo calcula en su `fit` | sí (aleatoria) |
| `lightgbm` | *Boosting* como XGBoost, pero haciendo crecer los árboles hoja a hoja, que es más rápido. | `class_weight="balanced"` | sí (aleatoria) |

La clase A tiene solo unos 24 tramos por ciudad, así que todos los modelos la compensan con pesos, y la métrica principal es la **F1 macro**, que da el mismo peso a cada clase.

## Qué es un hiperparámetro y cuáles ajustamos
Un **hiperparámetro** es un valor que se decide *antes* de entrenar y que el modelo no aprende de los datos (por ejemplo, cuánta profundidad pueden tener los árboles). **Ajustar** es elegir esos valores probando varias combinaciones y quedándonos con la que mejor funciona **con datos que el modelo no ha visto al entrenar**.

Esto último es lo importante. Si puntuáramos cada combinación con los mismos datos con los que se entrenó, ganaría la que más memoriza, no la que mejor generaliza. Es como un examen con las preguntas que ya conoces: sacar buena nota no demuestra que sepas la materia.

Por eso el orden es este:
1. **Ajustar:** probamos las combinaciones solo con las filas de entrenamiento, divididas en 3 pliegues. Cada combinación se entrena con 2 pliegues y se puntúa con el tercero, que no ha visto. Se repite 3 veces y se hace la media.
2. **Entrenar:** con la combinación ganadora, entrenamos el modelo final con todas las filas de entrenamiento.
3. **Evaluar:** puntuamos ese modelo una sola vez con el test, que no se ha usado en ningún paso anterior.

Los espacios de búsqueda están en `tuning_constants.py`. Aquí los leemos directamente del código:

In [ ]:
def flatten(space):
    """A search space as {hyperparameter: values}. A list of grids (logistic regression) is merged."""
    grids = space if isinstance(space, list) else [space]
    merged = {}
    for grid in grids:
        for name, values in grid.items():
            current = merged.setdefault(name, [])
            if isinstance(values, list):
                current.extend(v for v in values if v not in current)
            else:
                merged[name] = values  # a scipy distribution
    return merged


def describe(values):
    """The values of a hyperparameter as text."""
    if hasattr(values, "dist"):  # a scipy distribution
        name, args = values.dist.name, values.args
        if name == "randint":
            return f"entero entre {args[0]} y {args[1] - 1}"
        if name == "uniform":
            return f"decimal entre {args[0]} y {args[0] + args[1]:.1f}"
        if name == "loguniform":
            return f"decimal entre {args[0]} y {args[1]} (escala logarítmica)"
    return ", ".join(str(v) for v in values)


rows = []
for model, spec in SEARCH_SPACES.items():
    kind = "rejilla (todas las combinaciones)" if spec["kind"] == "grid" else f"aleatoria ({TUNING_N_CANDIDATES} candidatas)"
    for name, values in flatten(spec["space"]).items():
        rows.append({"modelo": model, "búsqueda": kind, "hiperparámetro": name, "valores que se prueban": describe(values)})
pd.DataFrame(rows)

### Qué significa cada hiperparámetro

**Regresión logística** ([documentación de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html))
- `C`: cuánto se deja libre al modelo. Un `C` pequeño lo obliga a ser simple (pesos pequeños), y uno grande le deja ajustarse mucho a los datos de entrenamiento.
- `penalty`: cómo se frena al modelo. `l2` reduce todos los pesos, y `l1` lleva algunos a cero, es decir, descarta variables.
- `solver`: el algoritmo que calcula los pesos. No cambia el modelo, solo cómo se calcula. `lbfgs` va con `l2` y `saga` con `l1`.

**Random forest** ([documentación de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html))
- `max_depth`: profundidad máxima de cada árbol, o cuántas preguntas seguidas puede hacer. Más profundo da más detalle, pero puede acabar memorizando.
- `min_samples_leaf`: mínimo de tramos que debe tener cada hoja final. Un valor alto da hojas más generales.
- `max_features`: fracción de variables que se consideran en cada división del árbol. Menos variables hace que los árboles sean más distintos entre sí.

**LightGBM** ([documentación de `LGBMClassifier`](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html)) y **XGBoost** ([documentación de `XGBClassifier`](https://xgboost.readthedocs.io/en/stable/python/python_api.html)): los dos construyen árboles uno detrás de otro
- `n_estimators`: número de árboles.
- `learning_rate`: cuánto corrige cada árbol nuevo a los anteriores. Si es pequeño aprende despacio y necesita más árboles.
- `subsample`: fracción de filas que usa cada árbol.
- `colsample_bytree`: fracción de variables que usa cada árbol.
- Solo LightGBM:
  - `num_leaves`: máximo de hojas por árbol. Más hojas da un árbol más complejo.
  - `min_child_samples`: mínimo de tramos por hoja, como `min_samples_leaf`.
- Solo XGBoost:
  - `max_depth`: profundidad máxima de cada árbol, como en el random forest.
  - `min_child_weight`: mínimo de peso que debe sumar una hoja. Es parecido a `min_child_samples`, pero cuenta el peso de los tramos, y los de clase A pesan más.

Valores que **no** se ajustan: `class_weight="balanced"` (se calcula a partir de los datos), `max_iter=1000` en la regresión logística y `n_estimators=300` en el random forest.

Dos detalles:
- En la regresión logística, `penalty` y `solver` van emparejados: `l1` solo funciona con `saga`, y `l2` con `lbfgs`.
- `min_child_samples` (LightGBM) y `min_samples_leaf` (random forest) tienen un tope bajo para evitar que hojas muy grandes hagan imposible predecir la clase A.

## Cómo se ajusta
El ajuste se hace dentro de cada una de las 17 particiones y solo con sus datos de entrenamiento. El test no se toca.

1. Dividimos el entrenamiento en 3 pliegues (A, B y C).
2. Para cada combinación de hiperparámetros:
   - entrenamos con B y C, y puntuamos con A
   - entrenamos con A y C, y puntuamos con B
   - entrenamos con A y B, y puntuamos con C

   La nota de la combinación es la media de las 3 puntuaciones (F1 macro).
3. Nos quedamos con la de mayor nota. Si hay otras a menos de 0,005, gana la más simple (indicamos el algoritmo de cuál es la más simple por cada modelo).
4. Con esa combinación entrenamos con todo el entrenamiento y evaluamos una sola vez en el test.

- Los **pliegues internos** (`StratifiedGroupKFold`) mantienen todas las filas de una misma calle en el mismo grupo, y reparten las clases de forma parecida en cada uno.
- El ajuste se hace con el conjunto de variables `completo` y el perfil `global`, y los valores elegidos se reutilizan para todos los conjuntos de variables y perfiles de esa partición.
- La **regla de desempate** elige lo más simple porque una diferencia tan pequeña puede deberse al ruido.

### Los pliegues internos de una partición
Para verlo con datos, repartimos el entrenamiento de una partición en 3 pliegues internos (validación cruzada anidada). Comprobamos que ninguna calle aparece a la vez en el entrenamiento y en la validación de un mismo pliegue.

In [ ]:
splits = build_all_splits(df)
split = splits[0]
indexed = df.set_index(ID_COL, drop=False)
X, y = get_xy(indexed, TUNING_FEATURE_SET, TUNING_PROFILE)
X_train, y_train = X.loc[split.train_ids], y.loc[split.train_ids]
streets = indexed.loc[split.train_ids, STREET_COL]

cv = StratifiedGroupKFold(n_splits=TUNING_INNER_SPLITS, shuffle=True, random_state=SEED)
rows = []
for i, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train, streets), start=1):
    rows.append({
        "pliegue interno": i,
        "filas de entrenamiento": len(train_idx),
        "filas de validación": len(val_idx),
        "tramos de clase A en la validación": int((y_train.iloc[val_idx] == 2).sum()),
        "calles en ambos": len(set(streets.iloc[train_idx]) & set(streets.iloc[val_idx])),
    })
print(f"Partición {split.key}: {len(split.train_ids)} filas de entrenamiento, {len(split.test_ids)} de test (no se tocan)")
pd.DataFrame(rows)

## Cómo se lanza el ajuste
`make tune` (o `python -m urban_accessibility_ml.modeling.tuning.tuning_run`) construye las 17 particiones, ajusta los 4 modelos en cada una y escribe los dos ficheros de los mejores hiperparámetros y todos los cálculos realizados. Solo sobreescribe resultados si se usa `--overwrite`.

Tarda bastantes minutos. El random forest y LightGBM son los más lentos.

## Resultados del ajuste

In [ ]:
print(f"{len(best_params)} particiones x {len(TUNED_MODELS)} modelos = {len(best_params) * len(TUNED_MODELS)} ajustes")
print(f"{len(candidates)} candidatas evaluadas, {int(candidates['selected'].sum())} elegidas (una por partición y modelo)")
candidates.groupby("model").size().rename("candidatas").to_frame()

### Qué hiperparámetros han sido elegidos por cada modelo
Vamos a mostrar por cada modelo, para cada una de las particiones, qué valores se han elegido de los hiperparámetros.
Como nota, cada partición se ha ajustado de manera independiente, por lo que los resultados pueden variar.

In [ ]:
chosen = {}
for model in TUNED_MODELS:
    chosen[model] = pd.DataFrame.from_dict({key: params[model] for key, params in best_params.items()}, orient="index")
    print(f"=== {model}")
    display(chosen[model].round(3))

### Regla de desempate
Vamos a mostrar, para ver en un ejemplo con el random forest, en cuántos casos se han elegido unos hiperparámetros más simples que los que tienen f1_macro más alto.

La columna `selected` marca la que se eligió, que **no tiene por qué ser la de nota más alta**: si otra candidata está a menos de 0,005 de la mejor, gana la más simple.

In [ ]:
example = candidates[
    (candidates["strategy"] == "within_city") & (candidates["name"] == "Albacete")
    & (candidates["fold"] == 0) & (candidates["model"] == "random_forest")
]
display(example.sort_values("f1_macro_mean", ascending=False).head(6)[["params", "f1_macro_mean", "f1_macro_std", "selected"]].round(4))

best_mean = candidates.groupby(PARTITION + ["model"])["f1_macro_mean"].transform("max")
chosen_rows = candidates[candidates["selected"]]
not_the_best = int((chosen_rows["f1_macro_mean"] < best_mean[chosen_rows.index]).sum())
print(f"En {not_the_best} de los {len(chosen_rows)} casos la candidata elegida no es la de mayor nota: la regla de desempate eligió una más simple.")

### Evolución del ajuste
Vamos a mostrar cómo ha evolucionado el valor de F1 macro en alguno de los hiperparámetros.

Empezamos mostrando la evolución del parámetro C en la regresión logística. 
Si todas las candidatas diesen casi la misma nota, ajustar apenas serviría. En esta figura, la nota de la regresión logística según `C` (cada línea es una partición).

**Importante:** la nota es la F1 macro en la validación **interna**. Se elige la mejor nota entre muchas, y solo sirve para comparar candidatas de un mismo modelo y partición, **no para comparar modelos entre sí**. El rendimiento real se mide en el test, en la matriz de experimentos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.6))

lr = candidates[candidates["model"] == "logistic_regression"].copy()
lr["C"] = lr["params"].map(lambda p: p["C"])
lr["penalty"] = lr["params"].map(lambda p: p["penalty"])
styles = {"l2": "-", "l1": "--"}
for penalty, group in lr.groupby("penalty"):
    for _, part in group.groupby(PARTITION):
        part = part.sort_values("C")
        ax.plot(part["C"], part["f1_macro_mean"], styles[penalty], color=MODEL_COLORS["logistic_regression"], alpha=0.25, linewidth=1)
    mean = group.groupby("C")["f1_macro_mean"].mean()
    ax.plot(mean.index, mean.values, styles[penalty], color=MODEL_COLORS["logistic_regression"], linewidth=2.5, label=f"{penalty} (media de las particiones)")
ax.set_xscale("log")
ax.set_xlabel("C (más pequeño = más regularización)")
ax.set_ylabel("F1 macro (validación interna)")
ax.set_title("Regresión logística")
ax.legend()
plt.tight_layout()
plt.show()

Vemos que al principio, a medida que va aumentando C, la F1 macro aumenta, luego se estabiliza.


## Cómo se usan los hiperparámetros
`run_one` y `run_experiments` reciben un parámetro `hyperparameters`: un diccionario `{clave de partición: {modelo: hiperparámetros}}`, que es justo lo que contiene `best_params.json`.

- Sin él (`None`), cada modelo se entrena con sus valores por defecto, como antes.
- Con él, `run_one` crea el modelo con esos hiperparámetros.

In [ ]:
print("Clave de la partición:", split.key)
for model, params in best_params[split.key].items():
    print(f"=== {model}")
    table = pd.DataFrame({"hiperparámetro": list(params), "valor": [str(round(v, 3)) if isinstance(v, float) else str(v) for v in params.values()]})
    display(table.style.hide(axis="index"))

## Resumen y siguientes pasos
- Se han ajustado cuatro modelos, en las 17 particiones y solo con sus datos de entrenamiento. Los resultados están en `reports/results/best_params.json` y `reports/results/tuning-candidates.csv`.
- Las notas del ajuste son de validación interna y no sirven para comparar modelos.
- Falta volver a lanzar la matriz de experimentos (notebook 10) pasando `hyperparameters=best_params`, para ver cuánto mejoran los modelos en el test frente a los valores por defecto.